# 13 - 每日信号顾问

逐票独立计算置信度(0-100)，输出多档操作建议。

**包含：**
1. 初始化
2. 分位数校准（首次/每月运行，生成每票的 confidence 阈值）
3. 历史信号回填（首次运行，后续自动跳过已有日期）
4. 生成今日操作建议
5. 操作建议表格 + 逐票归因
6. 个股走势 + 历史买卖信号标记
7. 写入数据库

## 1. 初始化

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.font_manager import fontManager, FontManager
import os
from datetime import datetime
from IPython.display import display, HTML

_cache_dir = matplotlib.get_cachedir()
for _f in os.listdir(_cache_dir):
    if _f.startswith('fontlist'):
        os.remove(os.path.join(_cache_dir, _f))
_fm = FontManager()
fontManager.__dict__.update(_fm.__dict__)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK JP', 'Noto Sans CJK SC', 'Droid Sans Fallback'] + plt.rcParams['font.sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 11

from invest_model.db import get_engine
from invest_model.repositories.stock_pool_repo import StockPoolRepository
from invest_model.repositories.stock_daily_repo import StockDailyRepository
from invest_model.repositories.etf_repo import ETFRepository
from invest_model.advisor import StockAdvisor
from invest_model.advisor.persistence import save_advisor_signals, get_advisor_history

engine = get_engine()
pool_repo = StockPoolRepository(engine)
daily_repo = StockDailyRepository(engine)
etf_repo = ETFRepository(engine)

# 合并 core + etf 两个池
_core_df = pool_repo.get_pool('core')
_etf_df = pool_repo.get_pool('etf')
_all_pool = pd.concat([_core_df, _etf_df], ignore_index=True)
all_codes = _all_pool['code'].tolist()
code_name_map = dict(zip(_all_pool['code'], _all_pool['name']))

# 获取信号日期：取数据库中已有数据的最新交易日
# (如果今天的数据还未通过 pipeline 拉取，则为 T-1)
latest_dates = []
for c in all_codes:
    d = daily_repo.get_latest_date(code=c)
    if not d:
        d = etf_repo.get_latest_date(c)
    if d:
        latest_dates.append(d)

trade_date = max(latest_dates) if latest_dates else datetime.now().strftime('%Y%m%d')
print(f'全池 {len(all_codes)} 只 (core {len(_core_df)} + etf {len(_etf_df)})')
print(f'信号日: {trade_date} (数据库最新已入库交易日，如需当天数据请先运行 daily_pipeline)')
print()
for c in all_codes:
    print(f'  {c} {code_name_map.get(c, "")}')

## 2. 分位数校准

基于每只标的的 2 年历史 composite 分布计算分位数 profile，
并通过目标出手频率（每票每年 15-20 次）反推 confidence 阈值。

**首次运行必需，后续每月重跑一次更新即可（已有数据会覆盖更新）。**

In [ ]:
TARGET_ACTIONS_PER_YEAR = 17  # 目标: 每票每年出手约 15-20 次

from invest_model.advisor.calibration import calibrate_with_frequency
from invest_model.advisor.persistence import (
    save_calibration_profiles, load_calibration_profiles,
    save_advisor_signals, get_advisor_history
)

profiles = calibrate_with_frequency(
    engine, all_codes, trade_date,
    target_actions_per_year=TARGET_ACTIONS_PER_YEAR, window=500
)

n_saved = save_calibration_profiles(engine, profiles)
print(f'校准完成: {len(profiles)} 只标的, 已保存 {n_saved} 条')
print()
for code, p in profiles.items():
    print(f'  {code} {code_name_map.get(code, ""):8s} | '
          f'P90={p.composite_p90:.4f} | threshold={p.action_threshold} | '
          f'window={p.window_days}天')

# 没有 profile 的标的(如 ETF 无历史 composite 数据)使用 fallback
no_profile = [c for c in all_codes if c not in profiles]
if no_profile:
    print(f'\n  [!] 以下标的无历史 composite 数据, 使用 fallback 映射: {no_profile}')

cal_repo = CalendarRepository(engine)
all_trade_dates = cal_repo.get_trade_dates('20200101', trade_date)

# 取最近 N 个交易日
backfill_dates = all_trade_dates[-BACKFILL_DAYS:]
print(f'回填区间: {backfill_dates[0]} ~ {backfill_dates[-1]} ({len(backfill_dates)} 个交易日)')

# 检查哪些日期已有数据（用第一只 code 做采样检测）
existing = get_advisor_history(engine, all_codes[:1], backfill_dates[0], backfill_dates[-1])
existing_dates = set(existing['trade_date'].unique()) if not existing.empty else set()
todo_dates = [d for d in backfill_dates if d not in existing_dates]
print(f'已有 {len(existing_dates)} 天, 待回填 {len(todo_dates)} 天')

if todo_dates:
    advisor_bf = StockAdvisor(engine)
    total_saved = 0
    for i, dt in enumerate(todo_dates):
        sigs = advisor_bf.advise_batch(all_codes, dt, code_name_map)
        n = save_advisor_signals(engine, sigs)
        total_saved += n
        if (i + 1) % 20 == 0 or i == len(todo_dates) - 1:
            print(f'  进度 {i+1}/{len(todo_dates)}, 累计写入 {total_saved} 条')
    print(f'\n回填完成! 共写入 {total_saved} 条顾问信号')
else:
    print('所有日期已有数据, 无需回填')

## 3. 历史信号回填

首次运行需要回填过去约 1 年的顾问信号，后续已有日期自动跳过。
如需强制重算（校准参数变更后），先手动清表：`DELETE FROM stock_advisor_signal`

In [ ]:
BACKFILL_DAYS = 250  # 约1年交易日

from invest_model.repositories.calendar_repo import CalendarRepository
from invest_model.advisor import StockAdvisor

cal_repo = CalendarRepository(engine)
all_trade_dates = cal_repo.get_trade_dates('20200101', trade_date)
backfill_dates = all_trade_dates[-BACKFILL_DAYS:]
print(f'回填区间: {backfill_dates[0]} ~ {backfill_dates[-1]} ({len(backfill_dates)} 个交易日)')

existing = get_advisor_history(engine, all_codes[:1], backfill_dates[0], backfill_dates[-1])
existing_dates = set(existing['trade_date'].unique()) if not existing.empty else set()
todo_dates = [d for d in backfill_dates if d not in existing_dates]
print(f'已有 {len(existing_dates)} 天, 待回填 {len(todo_dates)} 天')

if todo_dates:
    advisor_bf = StockAdvisor(engine, profiles=profiles)
    total_saved = 0
    for i, dt in enumerate(todo_dates):
        sigs = advisor_bf.advise_batch(all_codes, dt, code_name_map)
        n = save_advisor_signals(engine, sigs)
        total_saved += n
        if (i + 1) % 20 == 0 or i == len(todo_dates) - 1:
            print(f'  进度 {i+1}/{len(todo_dates)}, 累计写入 {total_saved} 条')
    print(f'\n回填完成! 共写入 {total_saved} 条')
else:
    print('所有日期已有数据, 无需回填')

## 4. 生成今日操作建议

In [ ]:
advisor = StockAdvisor(engine, profiles=profiles)
signals = advisor.advise_batch(all_codes, trade_date, code_name_map)
print(f'已生成 {len(signals)} 条操作建议\n')

for s in signals:
    marker = '>>>' if s.confidence >= 60 else '---'
    print(f'{marker} {s.code} {s.name:8s} | {s.action_cn:4s} | '
          f'置信度 {s.confidence:3d} | 仓位 {s.position_pct:.0%} | '
          f'综合 {s.composite:+.3f}')

## 5. 操作建议表格

In [ ]:
report_df = advisor.daily_report(all_codes, trade_date, code_name_map)
if report_df.empty:
    print('无数据')
else:
    display(report_df[['代码', '名称', '操作', '置信度', '仓位%', '综合分',
                       '技术', '基本面', '资金流', '情绪', '触发规则']])

    print('\n== 逐票归因 ==')
    for _, row in report_df.iterrows():
        print(f'\n[{row["代码"]} {row["名称"]}] {row["操作"]} (置信度{row["置信度"]})')
        print(f'  {row["归因"]}')

## 6. 个股走势 + 历史买卖信号标记

每只股票画近期走势，叠加历史操作建议的标记。

In [ ]:
CHART_DAYS = 120
from datetime import timedelta
from sqlalchemy import text as _text

# 确保表存在
with engine.begin() as _conn:
    _conn.execute(_text("""
        CREATE TABLE IF NOT EXISTS stock_advisor_signal (
            code VARCHAR(16) NOT NULL,
            trade_date VARCHAR(8) NOT NULL,
            action VARCHAR(16) NOT NULL,
            confidence INT NOT NULL,
            position_pct DECIMAL(5,4),
            composite DECIMAL(8,5),
            tech_score DECIMAL(8,5),
            fund_score DECIMAL(8,5),
            flow_score DECIMAL(8,5),
            sent_score DECIMAL(8,5),
            triggers JSON,
            attribution TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (code, trade_date),
            INDEX idx_trade_date (trade_date),
            INDEX idx_confidence (trade_date, confidence)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """))
print('stock_advisor_signal 表已就绪')

try:
    end_dt = datetime.strptime(trade_date, '%Y%m%d')
except ValueError:
    end_dt = datetime.now()
chart_start = (end_dt - timedelta(days=CHART_DAYS * 2)).strftime('%Y%m%d')

# 重新导入最新版本
import importlib, invest_model.advisor.persistence as _p
importlib.reload(_p)
from invest_model.advisor.persistence import get_advisor_history

hist = get_advisor_history(engine, all_codes, chart_start, trade_date)
n_stocks = len(all_codes)
fig, axes = plt.subplots(n_stocks, 1, figsize=(14, 4 * n_stocks), squeeze=False)
axes = axes.flatten()

for ax, code in zip(axes, all_codes):
    name = code_name_map.get(code, code)
    daily = daily_repo.get_daily(code, chart_start, trade_date)
    if daily.empty:
        daily = etf_repo.get_daily(code, chart_start, trade_date)
    if daily.empty:
        ax.set_title(f'{code} {name} - 无日线')
        continue
    daily['date'] = pd.to_datetime(daily['trade_date'], format='%Y%m%d')
    daily['close'] = pd.to_numeric(daily['close'], errors='coerce')
    daily = daily.tail(CHART_DAYS)
    ax.plot(daily['date'], daily['close'], color='#333', linewidth=0.8)

    if not hist.empty:
        code_hist = hist[hist['code'] == code].copy()
        if not code_hist.empty:
            code_hist['date'] = pd.to_datetime(code_hist['trade_date'], format='%Y%m%d')
            merged = code_hist.merge(daily[['date', 'close']], on='date', how='inner')

            buys = merged[merged['action'].isin(['strong_buy', 'buy'])]
            if not buys.empty:
                ax.scatter(buys['date'], buys['close'], marker='^', color='green',
                           s=80, zorder=5, label='买入')
            sells = merged[merged['action'].isin(['reduce', 'clear'])]
            if not sells.empty:
                ax.scatter(sells['date'], sells['close'], marker='v', color='red',
                           s=80, zorder=5, label='减仓/清仓')

    # 今日信号标注
    today_sig = next((s for s in signals if s.code == code), None)
    if today_sig and today_sig.action != 'hold':
        today_dt = pd.Timestamp(trade_date)
        today_price = daily.iloc[-1]['close']
        color = 'green' if today_sig.action in ('strong_buy', 'buy') else 'red'
        ax.annotate(f'{today_sig.action_cn}({today_sig.confidence})',
                    xy=(today_dt, today_price),
                    fontsize=9, color=color, fontweight='bold',
                    xytext=(10, 15), textcoords='offset points',
                    arrowprops=dict(arrowstyle='->', color=color))

    ax.set_title(f'{code} {name}', fontsize=12)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 7. 写入数据库（今日信号）

In [ ]:
try:
    n = save_advisor_signals(engine, signals)
    print(f'已保存 {n} 条顾问信号')
except Exception as e:
    print(f'保存失败: {e}')
    print('请确保已运行 01_database.ipynb 创建 stock_advisor_signal 表')